# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [1]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5024 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [2]:
import functools

# TODO: Implementacja dekoratora
def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Sprawdzanie argumentów pozycyjnych (*args)
        for arg in args:
            if isinstance(arg, list):
                print(f"Liczba elementów listy (args): {len(arg)}")

        # Sprawdzanie argumentów nazwanych (**kwargs)
        for value in kwargs.values():
            if isinstance(value, list):
                print(f"Liczba elementów listy (kwargs): {len(value)}")

        # Wywołanie oryginalnej funkcji
        return func(*args, **kwargs)
    return wrapper

# Test:
@show_list_length
def process_data(data_list, name, extra_list=None):
    print(f"Przetwarzanie {name}")

print("--- Test 1: Lista w args ---")
process_data([1, 2, 3, 4, 5], "Użytkownicy")

print("\n--- Test 2: Lista w kwargs ---")
process_data(data_list=["A", "B"], name="Produkty", extra_list=[10, 20, 30])

--- Test 1: Lista w args ---
Liczba elementów listy (args): 5
Przetwarzanie Użytkownicy

--- Test 2: Lista w kwargs ---
Liczba elementów listy (kwargs): 2
Liczba elementów listy (kwargs): 3
Przetwarzanie Produkty


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [3]:
import functools
from datetime import datetime
import time

# TODO: Implementacja dekoratora z argumentem
def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Rejestracja czasu startu i aktualnej daty
            start_time = time.time()
            current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Wywołanie oryginalnej funkcji
            result = func(*args, **kwargs)

            # Obliczenie czasu wykonania
            execution_time = time.time() - start_time

            # Przygotowanie wiadomości i zapis do pliku (tryb 'a' - append)
            log_message = f"[{current_date}] Funkcja: '{func.__name__}' | Czas wykonania: {execution_time:.6f} s\n"
            with open(filename, "a", encoding="utf-8") as file:
                file.write(log_message)

            return result
        return wrapper
    return decorator

@logger("execution_logs.log")
def heavy_computation(n):
    """Przykładowa funkcja symulująca obciążenie."""
    print(f"Rozpoczynam obliczenia dla n={n}...")
    total = 0
    for i in range(n):
        total += i
    time.sleep(0.5)  # Sztuczne opóźnienie, by czas wykonania był zauważalny
    print("Obliczenia zakończone.")
    return total

heavy_computation(1000000)
heavy_computation(5000000)

Rozpoczynam obliczenia dla n=1000000...
Obliczenia zakończone.
Rozpoczynam obliczenia dla n=5000000...
Obliczenia zakończone.


12499997500000

--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [4]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [5]:
# TODO: Implementacja deskryptora logującego dostęp
class AccessLogger:
    def __set_name__(self, owner, name):
        # Zapisujemy nazwę atrybutu (np. 'imie') oraz tworzymy nazwę
        # dla atrybutu prywatnego (np. '_imie'), w którym realnie przechowamy dane.
        self.public_name = name
        self.private_name = '_' + name

    def __get__(self, obj, objtype=None):
        # Odwyołanie na poziomie klasy (np. Uzytkownik.imie) zwraca samego deskryptora
        if obj is None:
            return self

        # Pobieramy wartość z prywatnego atrybutu instancji
        value = getattr(obj, self.private_name)
        print(f"[LOG] Odczyt atrybutu '{self.public_name}': {value}")
        return value

    def __set__(self, obj, value):
        print(f"[LOG] Zapis do atrybutu '{self.public_name}': {value}")
        # Zapisujemy wartość do prywatnego atrybutu instancji
        setattr(obj, self.private_name, value)


class Uzytkownik:
    # Przypisujemy deskryptory do atrybutów klasy
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        # Podczas inicjalizacji wywoła się już metoda __set__ deskryptora
        self.imie = imie
        self.wiek = wiek

print("--- Tworzenie użytkownika ---")
user = Uzytkownik("Jan", 30)

print("\n--- Odczyt atrybutów ---")
obecne_imie = user.imie
obecny_wiek = user.wiek

print("\n--- Zmiana wartości atrybutów ---")
user.imie = "Piotr"
user.wiek = 31

--- Tworzenie użytkownika ---
[LOG] Zapis do atrybutu 'imie': Jan
[LOG] Zapis do atrybutu 'wiek': 30

--- Odczyt atrybutów ---
[LOG] Odczyt atrybutu 'imie': Jan
[LOG] Odczyt atrybutu 'wiek': 30

--- Zmiana wartości atrybutów ---
[LOG] Zapis do atrybutu 'imie': Piotr
[LOG] Zapis do atrybutu 'wiek': 31


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [6]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [7]:
# TODO: Implementacja generatora ciągu Collatza
def collatz_generator(n):
    # Opcjonalne zabezpieczenie: Ciąg Collatza jest zdefiniowany dla liczb naturalnych dodatnich
    if n <= 0 or not isinstance(n, int):
        raise ValueError("Liczba początkowa musi być dodatnią liczbą całkowitą.")

    # Zwracamy pierwszą (początkową) wartość ciągu
    yield n

    # Generujemy kolejne wartości dopóki n nie osiągnie 1
    while n != 1:
        if n % 2 == 0:
            n = n // 2  # Dzielenie całkowitoliczbowe
        else:
            n = 3 * n + 1

        yield n

# Test:
for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [8]:
current_user = {"username": "admin", "role": "superuser"}

# TODO: Implementacja dekoratora @require_role
def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Pobieramy obecną rolę ze słownika (używając .get() dla bezpieczeństwa)
            user_role = current_user.get("role")

            # Weryfikacja uprawnień
            if user_role != role:
                raise PermissionError(
                    f"Brak dostępu! Wymagana rola to '{role}', a Twój profil posiada rolę '{user_role}'."
                )

            # Wywołanie funkcji w przypadku sukcesu autoryzacji
            return func(*args, **kwargs)
        return wrapper
    return decorator

# --- TEST ---

@require_role("superuser")
def delete_database():
    print("Baza danych została pomyślnie usunięta.")

@require_role("moderator")
def ban_user(username):
    print(f"Użytkownik {username} został zbanowany.")

print("--- Test 1: Pomyślna autoryzacja ---")
delete_database()  # Rola 'superuser' się zgadza

print("\n--- Test 2: Brak uprawnień (PermissionError) ---")
try:
    ban_user("spammer_123")  # Rola 'moderator' się nie zgadza
except PermissionError as e:
    print(f"Przechwycono wyjątek: {e}")

--- Test 1: Pomyślna autoryzacja ---
Baza danych została pomyślnie usunięta.

--- Test 2: Brak uprawnień (PermissionError) ---
Przechwycono wyjątek: Brak dostępu! Wymagana rola to 'moderator', a Twój profil posiada rolę 'superuser'.


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [11]:
# TODO: Implementacja deskryptora Typed
class Typed:
    def __init__(self, expected_type):
        # Zapisujemy oczekiwany typ (np. int, str) podczas tworzenia deskryptora
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        # Automatyczne pobranie nazwy atrybutu (od Pythona 3.6)
        self.public_name = name
        self.private_name = '_' + name

    def __get__(self, obj, objtype=None):
        # Obsługa odwołania bezpośrednio z klasy
        if obj is None:
            return self

        # Zwrócenie wartości ukrytej zmiennej
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        # Walidacja typu przed przypisaniem
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Wartość atrybutu '{self.public_name}' musi być typu "
                f"{self.expected_type.__name__}, a otrzymano {type(value).__name__}."
            )

        # Zapisanie poprawnej wartości do ukrytej zmiennej w instancji
        setattr(obj, self.private_name, value)

class Produkt:
    # Używamy deskryptora do wymuszenia konkretnych typów
    nazwa = Typed(str)
    cena = Typed(float)
    ilosc = Typed(int)

    def __init__(self, nazwa, cena, ilosc):
        self.nazwa = nazwa
        self.cena = cena
        self.ilosc = ilosc

telefon = Produkt("Smartfon", 1999.99, 5)
print(f"Dodano produkt: {telefon.nazwa}, cena: {telefon.cena}, ilość: {telefon.ilosc}")

Dodano produkt: Smartfon, cena: 1999.99, ilość: 5


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [12]:
# TODO: Implementacja generatora liczb pierwszych
def prime_generator():
    yield 2

    n = 3
    while True:
        jest_pierwsza = True
        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                jest_pierwsza = False
                break

        if jest_pierwsza:
            yield n

        n += 2

primes_ending_in_7 = (p for p in prime_generator() if p % 10 == 7)

licznik = 0
for prime in primes_ending_in_7:
    print(prime)
    licznik += 1
    if licznik == 10:
        break

7
17
37
47
67
97
107
127
137
157
